<a href="https://colab.research.google.com/github/camilofuentesac-afk/Ev1-Machine-Learning/blob/main/Evaluaci%C3%B3n_1_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluación 1 Machine Learning

Profesor: Franco Mansilla Ibañez

Fecha: 30/03/2026

Integrantes: Camilo Fuentes - Eduardo Lobos - Patricio Meza

### Introducción.
En la actualidad, el manejo de grandes volumenes de información basado en datos se han transformado en un aliado estrategico para la toma de decisiones en las diferentes organizaciones.

En en presente documento se va a desarrollar mediante a un rol de analista de datos y equipos de data science para abordar la resolución de un problema aplicado utilizando un dataset proveniente de Kaggle.

En primer lugar, se realizará una presentación de diferentes Framework para aplicar la analisitca de datos provenientes a pasos consecutivos de la descripción del mismo, limitaciones y alcances, para luego realizar una exploración de la información para mejorar la calidad de los datos realizando una limpieza, identificación, tratamientos de missing value y los atipicos.

Por consecuente, se dará a conocer las ventajas del desarrollo en conjunto a la validación y los test propuestos de los métodos para separar los datos de muestra de entramiento, valudación y los test fundamentando sus diferencias, ventajas y desventajas.

Para finalizar, se va a sintentizar toda información disponible aplicando un método de selección de la muestra de datos seleccionando e implementando el más adecuado según las caracteristicas del dataset y los objetivos propuestos desde el comienzo.

### 1) Realiza una presentación de diferentes Framework para poder aplicar analítica de datos.

Los Framework son herramientas especializadas o conjuntos de bibliotecas predefinidas que nos permiten el entrenamiento y despliegue de modelos de machine learning, evitando crear algoritmos o codigos complejos desde cero

1. Scikit-Learn (Modelamiento Estadístico y Machine Learning Clásico)
- Descripción: Es el framework estándar de la industria para implementar algoritmos de Machine Learning tradicional. Está construido sobre bases matemáticas sólidas (NumPy y SciPy) y ofrece un catálogo exhaustivo de modelos listos para ser entrenados.

- Alcances: Ideal para resolver problemas de Regresión (como predecir el precio de una casa) y Clasificación (como predecir supervivencia). Destaca por su API estandarizada: una vez que aprendes a entrenar un modelo, la sintaxis es idéntica para entrenar cualquier otro. Incluye además las herramientas perfectas para la evaluación de métricas y la partición de datos (como train_test_split o K-Fold).

- Limitaciones: No está diseñado para arquitecturas complejas como el Deep Learning (Redes Neuronales profundas). Tampoco está optimizado para funcionar en clústers de servidores distribuidos, limitando su uso a proyectos de tamaño pequeño a mediano que puedan procesarse en una sola máquina.

2. TensorFlow / Keras (Deep Learning e Inteligencia Artificial)
- Descripción: Desarrollado por Google, es un framework de código abierto de artillería pesada diseñado para el cálculo numérico complejo y la creación de redes neuronales artificiales profundas a gran escala.

- Alcances: Es el entorno natural para resolver problemas no estructurados de alta complejidad, como el reconocimiento de imágenes (visión computacional), procesamiento de lenguaje natural y modelos generativos. Permite aprovechar la aceleración por hardware utilizando tarjetas gráficas (GPUs) para reducir tiempos de entrenamiento masivos.

- Limitaciones: Posee una curva de aprendizaje sumamente empinada y requiere un alto poder computacional. Para problemas estructurados simples o de volumen moderado utilizar TensorFlow puede ser perjuicioso, esto porque genera modelos tipo "caja negra" que son muy difíciles de interpretar o explicar a la gerencia en comparación con un árbol de decisión de Scikit-Learn.

3. PyTorch (Framework de Deep Learning e Investigación Avanzada)
- Descripción: Desarrollado principalmente por el laboratorio de IA de Meta (Facebook), es un framework de código abierto basado en la biblioteca Torch, diseñado para ofrecer flexibilidad y velocidad en la creación de redes neuronales profundas mediante grafos de computación dinámica.

- Alcances: Es la herramienta líder en la investigación académica y está ganando la industria gracias a su diseño "Pythónico", que facilita la depuración de código. Es ideal para proyectos de visión computacional, procesamiento de lenguaje natural y modelos de aprendizaje por refuerzo, permitiendo una integración nativa con aceleración por GPU para procesar grandes volúmenes de datos.

- Limitaciones: Al estar orientado a la flexibilidad y al control total del desarrollador, puede requerir más líneas de código para tareas simples en comparación con librerías de alto nivel.



### 2) Calidad de los datos:

a. Realiza una exploración de los datos, explícalos.

In [ ]:
# Carga de librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ssl
from sklearn.model_selection import train_test_split
from scipy import stats

from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.metrics import f1_score, make_scorer

In [ ]:
# Carga de datos
ssl._create_default_https_context = ssl._create_unverified_context

url = "https://raw.githubusercontent.com/zeyongj/House-Prices-Advanced-Regression-Techniques/master/train.csv"

df = pd.read_csv(url)

print(f"El dataset tiene {df.shape[0]} filas (casas) y {df.shape[1]} columnas (variables).")
display(df.head())

In [ ]:
# Descripción de tipos de variables

# Identificación de vasriables numericas y categoricas
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

# Conteo de variables e identificación de estas
print(f"Total variables Numéricas: {len(num_cols)} | Total variables Categóricas: {len(cat_cols)}")
print("Numéricas:", num_cols)
print("Categóricas:", cat_cols)

# Descripción de variables numéricas
display(df[num_cols].describe().T)

# Resumen variables categoricas
display(df[cat_cols].describe().T)

b. Realiza una limpieza de datos.

In [ ]:
# identificación y tratamiento de datos atipicos
# 2.2 Verificación de datos atípicos y tratamiento mediante el metodo de precentiles (percentil 1 y 99)

def winsorize_p1_p99(df, cols):
    df_out = df.copy()
    bounds = {}
    for c in cols:
        p1 = df_out[c].quantile(0.01)
        p99 = df_out[c].quantile(0.99)
        bounds[c] = (p1, p99)
        df_out[c] = df_out[c].clip(lower=p1, upper=p99)
    return df_out, bounds

# Aplicar a nuestras variables numéricas
df_w, bounds = winsorize_p1_p99(df, num_cols)

# Reemplazamos nuestro dataframe original con el que ya está limpio
df = df_w.copy()

# Mostrar los límites calculados (p1, p99) por variable
bounds

In [ ]:
#Identificación y tratamiento de missing values
# conteo de missing values por variable
total_nulos = df.isnull().sum()

porcentaje_nulos = (total_nulos / len(df)) * 100

tabla_faltantes = pd.DataFrame({
    'Total Faltantes': total_nulos,
    'Porcentaje (%)': porcentaje_nulos
})

tabla_faltantes = tabla_faltantes[tabla_faltantes['Total Faltantes'] > 0].sort_values(by='Total Faltantes', ascending=False)

print("--- VARIABLES CON DATOS FALTANTES ---")
print()
display(tabla_faltantes)

In [ ]:
df_clean = df_w.copy()

# 1) Categóricas: Ciclo limpio (estilo profesor) con la etiqueta estructural
for c in cat_cols:
    df_clean[c] = df_clean[c].fillna("None")

# 2) Numéricas: Ciclo limpio, pero mejorado estadísticamente usando la Mediana
for c in num_cols:
    df_clean[c] = df_clean[c].fillna(df_clean[c].median())

# Verificar que ya no hay NA (mostramos el top 5 para confirmar)
print("--- VERIFICACIÓN DE NULOS ---")
display(df_clean.isna().sum().sort_values(ascending=False).head(10))
print(f"Total de datos nulos en todo el dataset: {df_clean.isna().sum().sum()}")

In [ ]:
# Verificación de correlaciones entre variables
corr = df_clean[num_cols].corr()


corr_abs = corr.abs()
#borrar la mitad de matriz
upper = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))


top_pairs = upper.stack().sort_values(ascending=False).head(15)
display(top_pairs)

# 2. Aislamos solo la columna del Precio de Venta (SalePrice)
# y la ordenamos de mayor a menor para ver las más fuertes arriba
correlacion_precio = corr['SalePrice'].sort_values(ascending=False)

# 3. Mostramos el Top 10 de las variables más importantes
# (Usamos .drop para quitar SalePrice vs SalePrice que siempre será 1.0)
print("--- TOP 10 VARIABLES CON MAYOR IMPACTO EN EL PRECIO ---")
display(correlacion_precio.drop('SalePrice').head(10))

### 3) Ventanas de desarrollo/validación/test:

Tipos de métodos para separar (Split) los datos en muestra de entrenamiento, validación y test.

1. Método Holdout Simple (Entrenamiento y Test)

Esta tecnica consiste en tomar el 100% de la base de datos original y dividirla aleatoreamente en subconjuntos mutuamente excluyentes.

Primer Subconjunto "Train": usualmente representa entre 80%-70% de los datos. Este se utiliza exclusivamente para alimentar el algoritmo y que este aprenda los patrones subyacentes.

Segundo subconjunto "Test": Este tambien llamdo conjunto de prueba representa entre el 30%-20% restante del conjunto de datos. Este porcentaje se le oculta al algoritmo durante la fase de aprendizaje y se utiliza al final para evaluar el rendimiento del modelo frente a datos que jamas se le han mostrado, esto para simular su comportamiento en el mundo real.

Ventajas:
- El metodo es mas facil de programar y ejecutarlo, generalmente este requiere solo una linea de codigo.
- Representa un bajo costo computacional, esto ya que el modelo solo debe entrenarse una unica vez, y ademas consume muy pocos recursos de memoria y tiempo de procesamiento, esto es ideal para prototipados rapidos.
- el modeolo es de una facil interpretación, esto ya que es muy intuitivo de explicar a personas no tecnicas.

Desventajas
- Alta varianza: omo el corte es aleatorio, por simple "mala suerte" todas las casas más caras podrían quedar en el grupo de prueba. Esto haría que el modelo se evalúe de forma injusta y los resultados varíen si cambias la "semilla" aleatoria.
- Pérdida de información para el aprendizaje: El 20% de los datos que apartas para el Test jamás se utilizan para entrenar. En bases de datos pequeñas, no usar esa información para enseñar al modelo es un lujo costoso.
- Riesgo de sobreajuste indirecto: Si ajustas tu modelo muchas veces intentando mejorar la nota que sacó en el conjunto de Test, indirectamente haces que el modelo "memorice" esos datos de prueba, perdiendo su capacidad de generalizar frente a datos totalmente nuevos.



2. Método de Partición en 3 Vías (Entrenamiento, Validación y Prueba)

Es un metodo similar al houdount, pero este consiste en tomar el 100% de la base de datos y dividirla en 3 subconjuntos distintos, como un ejemplo seria 60%-15%-15%.

Conjunto de Entrenamiento: La porción más grande, utilizada para que el algoritmo aprenda los patrones fundamentales de los datos.

Conjunto de Validación: Un subconjunto intermedio que funciona como un "examen de práctica". Se utiliza para comparar diferentes algoritmos entre sí o para afinar la configuración interna del modelo (ajuste de hiperparámetros) basándose en su rendimiento aquí.

Conjunto de Prueba: Son datos que se mantienen reservados hasta el final del proyecto. Solo se utilizan una vez para medir el rendimiento real e imparcial del modelo ya optimizado.

Ventajas:
- Evaluación imparcial: Al tener una ventana de validación separada, esta evita contaminar el conjunto de prueba. Asegura que la nota final del modelo sea completamente honesta y no el resultado de haberlo ajustado a la medida del examen.

- Permite la optimización: Es el entorno perfecto para experimentar.

Desventajas:
- Deficeincia de datos: Al dividir la muestra en tres partes, se reduce aún más la cantidad de ejemplos disponibles para que el modelo aprenda. En bases de datos pequeñas, esto puede hacer que el modelo no tenga suficiente información para encontrar buenos patrones.

- Mayor complejidad en el código: Requiere un poco más de trabajo de programación, ya que normalmente implica aplicar la función de división (train_test_split) dos veces consecutivas.

3. Metodo de Validación Cruzada (K-Fold)

Este metodo consiste en dividir el 100% de la base de datos en "k" partes o bloques del mismo tamaño. Es un proceso iterativo, esto se explica que, si se elije un k=5 la base de datos se dividira en 5 bloques y el modelo se va a entrenar 5 veces distintas desde cero. Por ejemplo en la primera ronda, usa 4 bloques para aprender y el bloque 1 para el examen. En la segunda ronda, usa otros 4 para aprender y el bloque 2 para el examen. Esto se repite hasta que todos los bloques hayan sido usados como examen exactamente una vez. La nota final del modelo es el promedio de los 5 exámenes.

Ventajas:
- Máximo aprovechamiento de los datos: A diferencia del Holdout, aquí el 100% de las filas de tu base de datos se utilizan tanto para entrenar como para evaluar. No se desperdicia ni un solo dato, lo cual es oro puro en bases de datos pequeñas.

- Evaluación ultra precisa (Baja varianza): Al promediar los resultados de múltiples exámenes distintos, eliminas por completo el factor "suerte" de cómo se dividieron los datos. Te da la métrica de rendimiento más realista y confiable posible.

Desventajas:

- Alto costo computacional: Como tienes que entrenar el algoritmo K veces completas, el tiempo de procesamiento y el uso de memoria de tu computador se multiplican por K. Si el modelo tarda 10 minutos en entrenar, con K=5 tardará 50 minutos.

- Mayor complejidad técnica: Requiere librerías y códigos un poco más avanzados para implementarlo y para extraer las conclusiones finales, ya que no terminas con un solo modelo, sino con el promedio de varios.

Metodo a elección

Método Holdout Simple (Entrenamiento y Test)

El método seleccionado para la ejecución de este primer modelo predictivo es el Método Holdout (Partición Simple 80/20). Esta decisión se fundamenta en la necesidad de establecer un modelo base eficiente y rápido. Dado que contamos con 1.460 registros, destinar el 80% al conjunto de entrenamiento proporciona un volumen de datos estadísticamente suficiente para que el algoritmo identifique los patrones del mercado inmobiliario, reservando un 20% intacto para una evaluación imparcial de su capacidad de generalización.

In [ ]:
# Aplicación del metodo seleccionado holdout 80/20
from sklearn.model_selection import train_test_split

# 1. Definir nuestra variable objetivo (lo que queremos predecir)
target = "SalePrice"

# 2. Separar la tabla en "Pistas" (X) y "Respuestas" (y)
X = df_clean.drop(columns=[target])
y = df_clean[target]

# 3. Aplicar el particionamiento Holdout (80% Entrenamiento, 20% Prueba)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,      # El 20% se guarda para el examen final
    random_state=42      # Semilla fija para que el corte sea replicable
)

# 4. Verificamos los tamaños para comprobar que no perdimos datos
print("--- TAMAÑOS DE LAS VENTANAS (HOLDOUT 80/20) ---")
print(f"Total de datos originales : X={X.shape}, y={y.shape}")
print(f"Entrenamiento (Train 80%) : X={X_train.shape}, y={y_train.shape}")
print(f"Prueba (Test 20%)         : X={X_test.shape}, y={y_test.shape}")

--- TAMAÑOS DE LAS VENTANAS (HOLDOUT 80/20) ---
Total de datos originales : X=(1460, 80), y=(1460,)
Entrenamiento (Train 80%) : X=(1168, 80), y=(1168,)
Prueba (Test 20%)         : X=(292, 80), y=(292,)


### Conclusión.
Tal como se ha logrado evidenciar a lo largo del presente documento, se logró construir un pipeline de analitica de datos abordando de una manera adecuada las etapas fundamentales a raiz de la selección de frameworks hasta la validación del modelo en lo que permitió obtener resultados favorables para facilitar la toma de decisiones tras el procesamiento del tipo de datos y los recursos disponibles.

El analisis compartivo de herramientas como Scikit-Learn, TensorFlow/Keras y PyTorch permitió visualizar que no existe una solución única para todos los problemas, sino que depende de otros factores como el nivel de complejidad o interpretaciones en el que se priorizó un enfoque adecuiado para los datos estructurado.

Importante a mencionar que, la limpieza de datos tiene un rol fundamental para obtener resultados confiables, en el que permite reducir sesgos y mejorar la estabilidad del modelo para evitar tomar una decisión erronea y potenciar la calidad de predicciones.

En síntesis, el módelo demuestra que en el analisis de datos no depende unicamente de un modelo utilizado, sino de un proceso que limpieza, procesamiento, y comprensión de información para que los datos sean de calidad para luego establecer metodologías de validación en la toma de decisiones.